# 逻辑回归（Logistic Regression）系统学习笔记

---
## 目录

1. [线性回归基础（前置知识）](#1-线性回归基础前置知识)
2. [从回归到分类：为什么需要逻辑回归](#2-从回归到分类为什么需要逻辑回归)
3. [Sigmoid 函数](#3-sigmoid-函数)
4. [逻辑回归的预测函数](#4-逻辑回归的预测函数)
5. [决策边界（Decision Boundary）](#5-决策边界decision-boundary)
6. [损失函数：从似然到交叉熵](#6-损失函数从似然到交叉熵)
7. [梯度推导与参数更新](#7-梯度推导与参数更新)
8. [多分类：Softmax 回归](#8-多分类softmax-回归)
9. [模型评估](#9-模型评估)
10. [实践要点与总结](#10-实践要点与总结)

---

## 1. 线性回归基础（前置知识）

逻辑回归虽然名字带"回归"，但它是一个**分类算法**。它的核心思想建立在线性回归之上，因此先回顾线性回归的关键概念。

### 1.1 问题形式化

以课件中的银行贷款为例：

| 工资（X₁） | 年龄（X₂） | 额度（Y） |
|:--------:|:--------:|:--------:|
| 4000     | 25       | 20000    |
| 8000     | 30       | 70000    |
| 5000     | 28       | 35000    |
| 7500     | 33       | 50000    |
| 12000    | 40       | 85000    |

- **特征（Feature）**：工资 X₁、年龄 X₂
- **标签（Label / Target）**：银行最终贷款额度 Y
- **目标**：找到特征与标签之间的映射关系

### 1.2 数学模型

线性回归假设特征与标签之间存在**线性关系**：

$$h_\theta(x) = \theta_0 + \theta_1 x_1 + \theta_2 x_2$$

- $\theta_1$：年龄的参数（权重），表示年龄每增加 1 岁，贷款额度变化的幅度
- $\theta_2$：工资的参数（权重），表示工资每增加 1 元，贷款额度变化的幅度
- $\theta_0$：偏置项（bias / intercept），基础额度

**向量化表示**（整合偏置项到特征向量中）：

$$h_\theta(x) = \theta^T x$$

其中 $x = [1, x_1, x_2]^T$（多加一列 1 对应 $\theta_0$），$\theta = [\theta_0, \theta_1, \theta_2]^T$。

### 1.3 误差项与高斯分布假设

真实值 $y^{(i)}$ 与预测值 $h_\theta(x^{(i)})$ 之间存在误差 $\varepsilon^{(i)}$：

$$y^{(i)} = \theta^T x^{(i)} + \varepsilon^{(i)}$$

对误差 $\varepsilon$ 的三个关键假设：

| 假设 | 含义 | 课件通俗解释 |
|:---:|------|------------|
| **独立性** | 各样本的误差相互独立 | 张三和李四一起来贷款，他俩没关系 |
| **同分布** | 所有样本来自同一分布 | 他俩都来的是同一家银行 |
| **高斯分布** | $\varepsilon \sim \mathcal{N}(0, \sigma^2)$，均值为 0 | 银行可能多给也可能少给，但绝大多数情况浮动不大 |

**为什么假设均值为 0？** 如果均值不为 0，说明模型存在系统性偏差（总是高估或低估），这个偏差可以被吸收进偏置项 $\theta_0$。

误差的概率密度函数：

$$p(\varepsilon^{(i)}) = \frac{1}{\sqrt{2\pi}\sigma} \exp\left(-\frac{(\varepsilon^{(i)})^2}{2\sigma^2}\right)$$

代入 $\varepsilon^{(i)} = y^{(i)} - \theta^T x^{(i)}$：

$$p(y^{(i)} \mid x^{(i)}; \theta) = \frac{1}{\sqrt{2\pi}\sigma} \exp\left(-\frac{(y^{(i)} - \theta^T x^{(i)})^2}{2\sigma^2}\right)$$

> 这条公式的含义：**给定特征 $x^{(i)}$ 和参数 $\theta$ 时，真实值 $y^{(i)}$ 出现的概率**。

### 1.4 似然函数与最大似然估计

**似然函数**：什么样的参数 $\theta$ 与我们的数据组合后，恰好能产生观测到的真实值？

$$L(\theta) = \prod_{i=1}^{m} p(y^{(i)} \mid x^{(i)}; \theta) = \prod_{i=1}^{m} \frac{1}{\sqrt{2\pi}\sigma} \exp\left(-\frac{(y^{(i)} - \theta^T x^{(i)})^2}{2\sigma^2}\right)$$

**对数似然**（乘法变加法，便于求导）：

$$\ell(\theta) = \ln L(\theta) = m \ln\frac{1}{\sqrt{2\pi}\sigma} - \frac{1}{2\sigma^2} \sum_{i=1}^{m} (y^{(i)} - \theta^T x^{(i)})^2$$

**核心洞察**：最大化 $\ell(\theta)$ 等价于最小化 $\sum_{i=1}^{m} (y^{(i)} - \theta^T x^{(i)})^2$——这就是**最小二乘法**的由来！

### 1.5 目标函数与求解

定义损失函数（Loss Function）：

$$J(\theta) = \frac{1}{2} \sum_{i=1}^{m} (h_\theta(x^{(i)}) - y^{(i)})^2$$

> 系数 $\frac{1}{2}$ 是为了求导时消去平方的系数，方便计算。

求偏导并令其等于 0，可得**解析解**（正规方程）：

$$\theta = (X^T X)^{-1} X^T y$$

但实际中，当特征维度高或 $X^T X$ 不可逆时，解析解不可行——这时就需要**梯度下降**。

### 1.6 梯度下降

核心思想：**一口吃不成胖子，每次优化一点点，累积起来就是大成绩。**

$$
\theta_j := \theta_j - \alpha \frac{\partial J(\theta)}{\partial \theta_j}
$$

其中 $\alpha$ 为**学习率（步长）**。

三种梯度下降变体：

| 方法 | 每次使用的数据量 | 优点 | 缺点 |
|:---|:---|------|------|
| **批量梯度下降（BGD）** | 全部样本 | 稳定收敛，易得最优解 | 速度慢，内存消耗大 |
| **随机梯度下降（SGD）** | 1 个样本 | 迭代速度快 | 震荡大，不一定每次朝收敛方向 |
| **小批量梯度下降（Mini-batch GD）** | 一小部分（如 32/64/128） | 兼顾速度与稳定性 | **最常用** |

### 1.7 模型评估：R²

$$R^2 = 1 - \frac{\sum_{i=1}^{m} (\hat{y}_i - y_i)^2}{\sum_{i=1}^{m} (\bar{y} - y_i)^2}$$

- 分子：残差平方和（SSE），模型未解释的变异
- 分母：总变异（SST），类似方差项
- $R^2 \in [0, 1]$，越接近 1 说明模型拟合越好

> $R^2$ 可以理解为：**模型解释了数据中多大比例的变异**。

---

## 2. 从回归到分类：为什么需要逻辑回归

### 2.1 线性回归做分类的问题

假设我们要预测"银行是否批准贷款"（是=1，否=0），如果直接用线性回归：

```
问题1：输出值可能 >1 或 <0，无法解释为概率
问题2：对异常值非常敏感，一个极端样本会大幅影响决策边界
问题3：线性回归假设误差服从高斯分布，分类任务不满足此假设
```

### 2.2 逻辑回归的核心思路

> **在线性回归的输出上套一层 Sigmoid 函数，将 $(-\infty, +\infty)$ 的实数映射到 $(0, 1)$ 区间，实现"值 → 概率"的转换。**

逻辑回归 = 线性回归 + Sigmoid 函数

---

## 3. Sigmoid 函数

### 3.1 公式与图像

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**关键性质：**

| 性质 | 说明 |
|:---:|:------:|
| 定义域 | $(-\infty, +\infty)$，任意实数 |
| 值域 | $(0, 1)$ |
| $\sigma(0)$ | $0.5$（分界点） |
| 对称性 | $\sigma(-z) = 1 - \sigma(z)$ |
| 单调性 | 严格单调递增 |
| 导数 | $\sigma'(z) = \sigma(z)(1 - \sigma(z))$（极其优雅，推导见下文） |

### 3.2 Sigmoid 函数的导数推导（重要！）

$$\begin{aligned}
\sigma'(z) &= \frac{d}{dz} \left( \frac{1}{1 + e^{-z}} \right) \\
&= \frac{e^{-z}}{(1 + e^{-z})^2} \\
&= \frac{1}{1 + e^{-z}} \cdot \frac{e^{-z}}{1 + e^{-z}} \\
&= \frac{1}{1 + e^{-z}} \cdot \left(1 - \frac{1}{1 + e^{-z}}\right) \\
&= \sigma(z)(1 - \sigma(z))
\end{aligned}$$

> 这个简洁的导数形式将在后续梯度推导中大大简化计算，是逻辑回归数学优美的关键之一。

---

## 4. 逻辑回归的预测函数

### 4.1 从线性回归到预测概率

设线性回归的输出为 $z = \theta^T x$，则逻辑回归的预测函数：

$$h_\theta(x) = \sigma(\theta^T x) = \frac{1}{1 + e^{-\theta^T x}}$$

**含义**：$h_\theta(x)$ 表示给定特征 $x$ 时，样本属于**正类（y=1）**的概率。

$$P(y=1 \mid x; \theta) = h_\theta(x)$$
$$P(y=0 \mid x; \theta) = 1 - h_\theta(x)$$

### 4.2 分类决策

通常以 **0.5** 为阈值：
- 若 $h_\theta(x) \geq 0.5$ → 预测为类别 1
- 若 $h_\theta(x) < 0.5$ → 预测为类别 0

> 阈值可以根据实际需求调整。例如医疗诊断中宁可假阳性多一些（降低阈值），也不能漏诊。

---

## 5. 决策边界（Decision Boundary）

### 5.1 线性决策边界

当 $h_\theta(x) = 0.5$ 时，$\theta^T x = 0$。因此：

$$\theta^T x = 0$$

就是**决策边界**——一个超平面（在二维空间中是一条直线）。

- $\theta^T x > 0$ 的区域 → 预测正类
- $\theta^T x < 0$ 的区域 → 预测负类

### 5.2 非线性决策边界

> 课件提到"逻辑回归的决策边界可以是非线性的"——这是很多人容易忽略的点。

通过在特征中加入多项式项，逻辑回归可以拟合非线性边界。例如：

$$h_\theta(x) = \sigma(\theta_0 + \theta_1 x_1 + \theta_2 x_2 + \theta_3 x_1^2 + \theta_4 x_2^2)$$

此时决策边界 $\theta_0 + \theta_1 x_1 + \theta_2 x_2 + \theta_3 x_1^2 + \theta_4 x_2^2 = 0$ 可以是圆形、椭圆形等曲线。

---

## 6. 损失函数：从似然到交叉熵

### 6.1 似然函数（分类版本）

对于二分类问题，每个样本的标签 $y^{(i)} \in \{0, 1\}$。综合两种情况：

$$P(y^{(i)} \mid x^{(i)}; \theta) = [h_\theta(x^{(i)})]^{y^{(i)}} \cdot [1 - h_\theta(x^{(i)})]^{1 - y^{(i)}}$$

> **巧妙的整合**：当 $y=1$ 时，后半部分指数为 0，整体只剩 $h_\theta(x)$；当 $y=0$ 时，前半部分指数为 0，整体只剩 $1 - h_\theta(x)$。

**似然函数**（全部 m 个样本的联合概率）：

$$L(\theta) = \prod_{i=1}^{m} [h_\theta(x^{(i)})]^{y^{(i)}} \cdot [1 - h_\theta(x^{(i)})]^{1 - y^{(i)}}$$

### 6.2 对数似然

$$\begin{aligned}
\ell(\theta) &= \ln L(\theta) \\
&= \sum_{i=1}^{m} \left[ y^{(i)} \ln h_\theta(x^{(i)}) + (1 - y^{(i)}) \ln (1 - h_\theta(x^{(i)})) \right]
\end{aligned}$$

### 6.3 交叉熵损失函数

我们的目标是**最大化**对数似然 $\ell(\theta)$。为了统一为"最小化损失"的优化框架，取负号并除以样本数：

$$J(\theta) = -\frac{1}{m} \ell(\theta) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \ln h_\theta(x^{(i)}) + (1 - y^{(i)}) \ln (1 - h_\theta(x^{(i)})) \right]$$

这就是大名鼎鼎的**交叉熵损失函数（Cross-Entropy Loss / Log Loss）**。

> 课件中提到"此时应用梯度上升求最大值，引入 $J(\theta)$ 转换为梯度下降任务"——正是通过取负号实现的。

### 6.4 为什么不用均方误差（MSE）？

如果用 MSE 作为逻辑回归的损失函数：

$$J_{\text{MSE}}(\theta) = \frac{1}{m} \sum_{i=1}^{m} (h_\theta(x^{(i)}) - y^{(i)})^2$$

由于 Sigmoid 的非线性，这个函数**非凸**，会有很多局部最优解，梯度下降容易陷入局部极小值。而交叉熵损失函数是**凸函数**，保证能找到全局最优解。

---

## 7. 梯度推导与参数更新

### 7.1 单个样本的梯度（核心推导）

先求 Sigmoid 函数对参数 $\theta_j$ 的偏导：

$$\frac{\partial h_\theta(x)}{\partial \theta_j} = h_\theta(x)(1 - h_\theta(x)) \cdot x_j$$

> 这里用到了前面 Sigmoid 导数的优雅性质。

现在求损失函数对 $\theta_j$ 的偏导。先看单个样本 $(x, y)$：

$$\begin{aligned}
\frac{\partial}{\partial \theta_j} \left[ y \ln h_\theta(x) + (1-y) \ln(1 - h_\theta(x)) \right]
&= y \cdot \frac{1}{h_\theta(x)} \cdot \frac{\partial h_\theta(x)}{\partial \theta_j} + (1-y) \cdot \frac{-1}{1 - h_\theta(x)} \cdot \frac{\partial h_\theta(x)}{\partial \theta_j} \\
&= \left[ \frac{y}{h_\theta(x)} - \frac{1-y}{1 - h_\theta(x)} \right] \cdot h_\theta(x)(1 - h_\theta(x)) \cdot x_j \\
&= \left[ y(1 - h_\theta(x)) - (1-y)h_\theta(x) \right] \cdot x_j \\
&= (y - h_\theta(x)) \cdot x_j
\end{aligned}$$

**惊艳的结果**：梯度的形式与线性回归惊人地相似——都是 **（预测值 - 真实值）× 特征**。

### 7.2 所有样本的梯度

$$\frac{\partial J(\theta)}{\partial \theta_j} = \frac{1}{m} \sum_{i=1}^{m} (h_\theta(x^{(i)}) - y^{(i)}) \cdot x_j^{(i)}$$

**向量化形式**：

$$\nabla J(\theta) = \frac{1}{m} X^T (h_\theta(X) - y)$$

> 线性回归的梯度：$\frac{1}{m} X^T (X\theta - y)$  
> 逻辑回归的梯度：$\frac{1}{m} X^T (\sigma(X\theta) - y)$  
> 唯一的区别是逻辑回归多了一层 Sigmoid 变换！

### 7.3 参数更新公式

$$\theta_j := \theta_j - \alpha \cdot \frac{1}{m} \sum_{i=1}^{m} (h_\theta(x^{(i)}) - y^{(i)}) \cdot x_j^{(i)}$$

向量化：

$$\theta := \theta - \frac{\alpha}{m} X^T (\sigma(X\theta) - y)$$

---

## 8. 多分类：Softmax 回归

> 课件提到：逻辑回归扩展为多分类时使用 Softmax。

### 8.1 Softmax 函数

对于 K 个类别，每个类别有自己的一套参数 $\theta_k$：

$$P(y=k \mid x; \theta) = \frac{e^{\theta_k^T x}}{\sum_{j=1}^{K} e^{\theta_j^T x}}$$

- 分子：类别 k 的"得分"取指数
- 分母：所有类别的指数得分之和（归一化因子）
- 输出：K 个概率值，和为 1

### 8.2 交叉熵损失（多分类版）

$$J(\theta) = -\frac{1}{m} \sum_{i=1}^{m} \sum_{k=1}^{K} \mathbb{1}\{y^{(i)} = k\} \ln P(y=k \mid x^{(i)}; \theta)$$

其中 $\mathbb{1}\{\cdot\}$ 是指示函数——只有当样本真实类别为 k 时才为 1。

### 8.3 与二分类逻辑回归的关系

当 K=2 时，Softmax 回归退化为逻辑回归。令 $\theta = \theta_1 - \theta_0$，有：

$$P(y=1 \mid x) = \frac{e^{\theta_1^T x}}{e^{\theta_0^T x} + e^{\theta_1^T x}} = \frac{1}{1 + e^{-(\theta_1 - \theta_0)^T x}} = \sigma(\theta^T x)$$

---

## 9. 模型评估

### 9.1 分类问题的评估指标

| 指标 | 公式 | 含义 |
|:---|:---|------|
| **准确率（Accuracy）** | $\frac{TP+TN}{TP+TN+FP+FN}$ | 预测正确的比例 |
| **精确率（Precision）** | $\frac{TP}{TP+FP}$ | 预测为正的样本中，真正为正的比例 |
| **召回率（Recall）** | $\frac{TP}{TP+FN}$ | 真正为正的样本中，被正确识别的比例 |
| **F1-Score** | $2 \cdot \frac{P \cdot R}{P + R}$ | 精确率和召回率的调和平均 |
| **AUC-ROC** | 曲线下面积 | 衡量模型区分正负类的能力（不受阈值影响） |

### 9.2 混淆矩阵

|  | 预测为正 | 预测为负 |
|:---|:---:|:---:|
| **实际为正** | TP | FN |
| **实际为负** | FP | TN |

---

## 10. 实践要点与总结

### 10.1 课件强调的核心观点

1. **先逻辑回归再复杂模型**："机器学习算法选择：先用逻辑回归，能简单的还是用简单的。"——这是工业界的黄金法则。逻辑回归作为基线模型（baseline），训练快、可解释性强。

2. **逻辑回归真的很好用**：速度快、占用内存少、可解释、易于部署，是工业界最常见模型之一。

### 10.2 学习率选择

- 从小开始尝试（如 0.01, 0.001）
- 太小：收敛慢
- 太大：震荡甚至发散
- 自适应方法：Adam、RMSprop 等可自动调整

### 10.3 批处理大小

- 常用值：32、64、128
- 考虑因素：内存容量和计算效率
- 一般 2 的幂次方可以更好利用 GPU 硬件

### 10.4 特征工程对逻辑回归至关重要

由于逻辑回归是线性分类器（在原始特征空间），特征工程直接影响模型能力上限：
- **特征缩放 / 归一化**：保证各特征尺度一致，梯度下降收敛更快
- **多项式特征**：增加非线性表达能力
- **特征交叉**：捕捉特征之间的交互效应
- **离散化 / 分箱**：处理非线性关系

### 10.5 正则化

为防止过拟合，通常加入 L1 或 L2 正则项：

- **L2 正则化（Ridge）**：$J(\theta) + \frac{\lambda}{2m} \sum_{j=1}^{n} \theta_j^2$
- **L1 正则化（Lasso）**：$J(\theta) + \frac{\lambda}{m} \sum_{j=1}^{n} |\theta_j|$（可产生稀疏解，做特征选择）

### 10.6 逻辑回归 vs 线性回归 对比总结

| 维度 | 线性回归 | 逻辑回归 |
|:---|:---|:---|
| **任务** | 回归（预测连续值） | 分类（预测离散类别） |
| **输出** | $(-\infty, +\infty)$ | $(0, 1)$，可解释为概率 |
| **预测函数** | $h_\theta(x) = \theta^T x$ | $h_\theta(x) = \sigma(\theta^T x)$ |
| **损失函数** | MSE（均方误差） | 交叉熵（Cross-Entropy） |
| **误差假设** | 高斯分布 | 伯努利分布（二分类） |
| **梯度形式** | $\frac{1}{m}X^T(X\theta - y)$ | $\frac{1}{m}X^T(\sigma(X\theta) - y)$ |
| **求解方法** | 有解析解（正规方程） | 无解析解，用梯度下降 |
| **损失函数凸性** | 凸 | 凸（交叉熵） / 非凸（MSE） |

### 10.7 一句话总结

> **逻辑回归 = 线性回归 + Sigmoid + 交叉熵。把线性模型的输出通过 Sigmoid 映射为概率，用最大化似然（等价于最小化交叉熵）来求解参数，得到的是一个简单、可解释、高效的概率分类器。**

---

## 附录：关键公式速查

| 名称 | 公式 |
|:---|:---|
| Sigmoid 函数 | $\sigma(z) = \dfrac{1}{1 + e^{-z}}$ |
| Sigmoid 导数 | $\sigma'(z) = \sigma(z)(1 - \sigma(z))$ |
| 预测函数 | $h_\theta(x) = \sigma(\theta^T x)$ |
| 似然函数（单样本） | $P(y \mid x) = h_\theta(x)^y \cdot (1 - h_\theta(x))^{1-y}$ |
| 损失函数（交叉熵） | $J(\theta) = -\frac{1}{m}\sum_{i=1}^{m}[y^{(i)}\ln h_\theta(x^{(i)}) + (1-y^{(i)})\ln(1-h_\theta(x^{(i)}))]$ |
| 梯度 | $\frac{\partial J}{\partial \theta_j} = \frac{1}{m}\sum_{i=1}^{m}(h_\theta(x^{(i)}) - y^{(i)})x_j^{(i)}$ |
| 参数更新 | $\theta := \theta - \frac{\alpha}{m}X^T(\sigma(X\theta) - y)$ |
| Softmax | $P(y=k \mid x) = \dfrac{e^{\theta_k^T x}}{\sum_{j=1}^{K} e^{\theta_j^T x}}$ |